# Exploración del dataset de mi proyecto

Proyecto: **GrantMatch** (Subvenciones con Ruta). Archivo explorado: `../data/processed/obt_award_work_muestra.csv`, una muestra de 50 000 filas de `OBT.OBT_AWARD_WORK`, la One Big Table que deja el pipeline en Snowflake. Se exporta con [`exportar_obt.sql`](./exportar_obt.sql).

Cada fila es un par (award, publicación) en el que el award financió la publicación. Este notebook es el registro de la exploración: el código, lo que mostró y lo que concluyo, en ese orden.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

datos = pd.read_csv("../data/processed/obt_award_work_muestra.csv")
datos.columns = datos.columns.str.lower()  # Snowflake exporta en MAYUSCULAS

# El modelo predice si un par (award, publicacion) es pertinente.
OBJETIVO = "label"

## Paso 1: resumen

In [ ]:
print("Filas y columnas:", datos.shape)
print(datos.dtypes)
print(datos.isna().sum())
datos.describe().round(2)

## Paso 2: distribución de la variable objetivo

In [ ]:
plt.figure()
plt.hist(datos[OBJETIVO], bins=40)
plt.title(OBJETIVO)
plt.show()
print(f"Valores distintos de {OBJETIVO}:", datos[OBJETIVO].value_counts().to_dict())

## Paso 3: atípicos de la variable objetivo con la regla del IQR

In [ ]:
q1 = datos[OBJETIVO].quantile(0.25)
q3 = datos[OBJETIVO].quantile(0.75)
iqr = q3 - q1
fuera = datos[(datos[OBJETIVO] < q1 - 1.5 * iqr) | (datos[OBJETIVO] > q3 + 1.5 * iqr)]
print(f"Atipicos de {OBJETIVO} segun IQR: {len(fuera)}")
fuera.head(10)

## Paso 4: la relación entre el objetivo y la variable que más esperas que influya

La señal que espero que más pese en la pertinencia es que el award y la publicación sean del mismo campo. Como en la OBT solo hay pares reales, no se puede promediar el objetivo por grupo (siempre da 1): miro qué porcentaje de pares reales coincide en campo y en dominio.

In [ ]:
datos["mismo_campo"] = datos["award_field_name"] == datos["work_field_name"]
datos["mismo_dominio"] = datos["award_domain_name"] == datos["work_domain_name"]
print("Pares reales con el mismo campo:", round(100 * datos["mismo_campo"].mean(), 1), "%")
print("Pares reales con el mismo dominio:", round(100 * datos["mismo_dominio"].mean(), 1), "%")

por_campo = (datos.groupby("award_field_name")["mismo_campo"]
             .agg(pares="size", pct_mismo_campo="mean")
             .sort_values("pares", ascending=False).head(15))
por_campo["pct_mismo_campo"] = (100 * por_campo["pct_mismo_campo"]).round(1)

plt.figure(figsize=(8, 6))
plt.barh(por_campo.index[::-1], por_campo["pct_mismo_campo"][::-1])
plt.xlabel("% de pares reales con el mismo campo")
plt.title("Coincidencia de campo award-publicacion")
plt.tight_layout()
plt.show()
por_campo

## Paso 5: tres hallazgos que afectarían el modelado

Para cada hallazgo: una celda de código con la consulta o el gráfico que lo confirma, y debajo una celda de texto con qué encontré, el número que lo confirma y qué cambia en el modelo por su causa.

### Hallazgo 1

In [ ]:
print("Proporcion de label = 1:", datos[OBJETIVO].mean())
print("Valores distintos:", datos[OBJETIVO].nunique())

**El objetivo es constante.** `label` vale 1 en las ⟨X⟩ filas: el job de Spark lo pone con `F.lit(1)` porque la OBT solo guarda financiamientos reales. Con un objetivo constante no hay nada que aprender. **Qué cambia:** antes de modelar hay que generar pares negativos (award + publicación que no financió), lo que convierte el problema en una clasificación balanceada. Lo hago en `features_proyecto.ipynb`.

### Hallazgo 2

In [ ]:
pares_por_award = datos["award_key"].value_counts()
top_1pct = pares_por_award.head(max(1, len(pares_por_award) // 100)).sum()
print(f"Awards distintos: {len(pares_por_award)}")
print(f"Pares del 1% de awards con mas publicaciones: {100 * top_1pct / len(datos):.1f}%")
print((100 * datos["funder_name"].value_counts(normalize=True).head(10)).round(1))

plt.figure()
plt.hist(datos["funded_outputs_count"].dropna(), bins=50, log=True)
plt.xlabel("Publicaciones financiadas por el award")
plt.ylabel("Pares (escala log)")
plt.title("Concentracion por award")
plt.show()

**Pocos awards y funders acumulan muchos pares.** El 1 % de awards con más publicaciones concentra ⟨X⟩ % de los pares, y el funder más frecuente, ⟨nombre⟩, ⟨X⟩ %. **Qué cambia:** si divido al azar, el mismo award aparece en entrenamiento y en prueba y el modelo se evalúa con convocatorias que ya conoce. Hay que dividir de forma que un award quede de un solo lado: por fecha de inicio del award. Además, `funded_outputs_count` se calcula con los resultados del financiamiento, así que no puede ser feature.

### Hallazgo 3

In [ ]:
sin_descripcion = 100 * (1 - datos["has_description"].astype(str).str.lower().eq("true").mean())
sin_abstract = 100 * (1 - datos["has_abstract"].astype(str).str.lower().eq("true").mean())
print(f"Pares sin descripcion del award: {sin_descripcion:.1f}%")
print(f"Pares sin abstract: {sin_abstract:.1f}%")
print(f"amount_usd vacio: {100 * datos['amount_usd'].isna().mean():.1f}%")
print("Montos marcados invalidos:", datos["is_amount_invalid"].astype(str).str.lower().eq("true").sum())
datos[["award_text_len", "work_text_len", "amount_usd"]].describe().round(0)

**Falta texto y falta monto en buena parte de los pares.** ⟨X⟩ % de los awards no tiene descripción, ⟨X⟩ % de las publicaciones no tiene abstract y ⟨X⟩ % de los montos está vacío. **Qué cambia:** un modelo que dependa del texto (embeddings) no sirve para esos pares, así que el primer modelo usa columnas estructuradas; `amount_usd` necesita imputación (mediana del entrenamiento) y escalado, porque su distribución tiene una cola muy larga.